# Retos del workshop RAG · William Villa

C1 ejecutado aquí. C4 y C5 se ejecutaron dentro del notebook 03 (reutilizan su índice y su retriever).

## C1 · HNSW contra fuerza bruta con FAISS

In [1]:
%pip install -q faiss-cpu matplotlib

In [2]:
import time
import faiss
import numpy as np

d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time(); index.add(xb); build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
        rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))

print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

exacto: 0.168 ms/consulta
M= 8 ef= 10 recall@10=0.168  0.015 ms/q  (build 1.7s)
M= 8 ef= 16 recall@10=0.242  0.024 ms/q  (build 1.7s)
M= 8 ef= 32 recall@10=0.355  0.027 ms/q  (build 1.7s)
M= 8 ef= 64 recall@10=0.510  0.055 ms/q  (build 1.7s)
M= 8 ef=128 recall@10=0.683  0.086 ms/q  (build 1.7s)
M= 8 ef=256 recall@10=0.819  0.176 ms/q  (build 1.7s)
M=16 ef= 10 recall@10=0.329  0.019 ms/q  (build 2.4s)
M=16 ef= 16 recall@10=0.425  0.036 ms/q  (build 2.4s)
M=16 ef= 32 recall@10=0.599  0.035 ms/q  (build 2.4s)
M=16 ef= 64 recall@10=0.770  0.055 ms/q  (build 2.4s)
M=16 ef=128 recall@10=0.905  0.106 ms/q  (build 2.4s)
M=16 ef=256 recall@10=0.971  0.276 ms/q  (build 2.4s)
M=32 ef= 10 recall@10=0.468  0.019 ms/q  (build 3.1s)
M=32 ef= 16 recall@10=0.579  0.024 ms/q  (build 3.1s)
M=32 ef= 32 recall@10=0.761  0.041 ms/q  (build 3.1s)
M=32 ef= 64 recall@10=0.909  0.083 ms/q  (build 3.1s)
M=32 ef=128 recall@10=0.983  0.155 ms/q  (build 3.1s)
M=32 ef=256 recall@10=0.998  0.313 ms/q  (build 3.1s)


In [3]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for M in (8, 16, 32):
    r = [x for x in rows if x[0] == M]
    ax[0].plot([x[1] for x in r], [x[2] for x in r], marker="o", label=f"M={M}")
    ax[1].plot([x[3] for x in r], [x[2] for x in r], marker="o", label=f"M={M}")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("efSearch"); ax[0].set_ylabel("recall@10"); ax[0].set_title("Recall vs efSearch (n=20.000)")
ax[1].axvline(t_flat, color="gray", ls="--", label="exacto"); ax[1].set_xlabel("ms por consulta"); ax[1].set_ylabel("recall@10"); ax[1].set_title("Recall vs latencia")
for a in ax: a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

> La gráfica generada por la celda anterior se reproduce en `RESPUESTAS.md` (sección C1) como gráfico Mermaid para que se vea directamente en GitHub.

In [4]:
# ¿Desde qué n HNSW (M=16) es más rápido que el exacto?
def experimento_n(n, M=16, efs=(16, 32, 64, 128, 256)):
    xb = rng.random((n, d), dtype="float32")
    flat = faiss.IndexFlatL2(d); flat.add(xb)
    t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000
    idx = faiss.IndexHNSWFlat(d, M); idx.hnsw.efConstruction = 100
    t0 = time.time(); idx.add(xb); build = time.time() - t0
    out = []
    for ef in efs:
        idx.hnsw.efSearch = ef
        t0 = time.time(); _, I = idx.search(xq, k); ms = (time.time() - t0) / nq * 1000
        out.append((ef, recall_at_k(I, I_true), ms))
    return t_flat, build, out

tabla_n = []
for n in (20_000, 100_000, 500_000):
    t_flat_n, build, out = experimento_n(n)
    print(f"n={n:>7}: exacto {t_flat_n:.3f} ms/q | HNSW build {build:.1f}s")
    for ef, rec, ms in out:
        print(f"   ef={ef:>3} recall={rec:.3f} {ms:.3f} ms/q  {'(más rápido que exacto)' if ms < t_flat_n else ''}")
    tabla_n.append((n, t_flat_n, build, out))

n=  20000: exacto 0.239 ms/q | HNSW build 2.6s
   ef= 16 recall=0.435 0.031 ms/q  (más rápido que exacto)
   ef= 32 recall=0.596 0.083 ms/q  (más rápido que exacto)
   ef= 64 recall=0.773 0.075 ms/q  (más rápido que exacto)
   ef=128 recall=0.908 0.134 ms/q  (más rápido que exacto)
   ef=256 recall=0.972 0.347 ms/q  


n= 100000: exacto 1.954 ms/q | HNSW build 19.1s
   ef= 16 recall=0.242 0.046 ms/q  (más rápido que exacto)
   ef= 32 recall=0.378 0.076 ms/q  (más rápido que exacto)
   ef= 64 recall=0.544 0.126 ms/q  (más rápido que exacto)
   ef=128 recall=0.701 0.211 ms/q  (más rápido que exacto)
   ef=256 recall=0.844 0.390 ms/q  (más rápido que exacto)


n= 500000: exacto 11.366 ms/q | HNSW build 130.8s
   ef= 16 recall=0.133 0.061 ms/q  (más rápido que exacto)
   ef= 32 recall=0.223 0.100 ms/q  (más rápido que exacto)
   ef= 64 recall=0.325 0.173 ms/q  (más rápido que exacto)
   ef=128 recall=0.464 0.315 ms/q  (más rápido que exacto)
   ef=256 recall=0.615 0.638 ms/q  (más rápido que exacto)
